In [1]:
# Install required packages
!pip install beautifulsoup4 requests pandas selenium webdriver-manager
!pip install openai python-dotenv
!pip install gspread oauth2client  # for Google Sheets integration

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [2]:
# Import required libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from datetime import datetime
import json
import re

# For web automation if needed
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# PROJECT PLAN
print("=== AI Lead Generation Agent ===")
print()
print("Target: Southeast Asian Startups")
print("Data Sources We'll Use:")
print("1. Crunchbase (public data)")
print("2. AngelList/Wellfound (startup listings)")
print("3. Tech startup directories")
print()
print("Data Points to Collect:")
print("- Company name")
print("- Industry/sector")
print("- Location")
print("- Website")
print("- Employee count (if available)")
print("- Contact email (if available)")
print("- Funding stage")

=== AI Lead Generation Agent ===

Target: Southeast Asian Startups
Data Sources We'll Use:
1. Crunchbase (public data)
2. AngelList/Wellfound (startup listings)
3. Tech startup directories

Data Points to Collect:
- Company name
- Industry/sector
- Location
- Website
- Employee count (if available)
- Contact email (if available)
- Funding stage


In [4]:
# Test our scraping setup
def test_scraping():
    url = "https://httpbin.org/html"  # Safe test URL
    response = requests.get(url)

    if response.status_code == 200:
        print("✅ Web scraping setup working!")
        print(f"Response length: {len(response.text)} characters")
    else:
        print("❌ Something went wrong")

test_scraping()

✅ Web scraping setup working!
Response length: 3739 characters


In [5]:
# Debug the test scraping function
def debug_test_scraping():
    url = "https://httpbin.org/html"
    print(f"Testing URL: {url}")

    try:
        response = requests.get(url, timeout=10)
        print(f"Status Code: {response.status_code}")
        print(f"Response length: {len(response.text)} characters")

        if response.status_code == 200:
            print("✅ Success!")
        else:
            print(f"❌ HTTP Error: {response.status_code}")
            print("Common codes:")
            print("  403 = Forbidden")
            print("  404 = Not Found")
            print("  500 = Server Error")

    except requests.exceptions.RequestException as e:
        print(f"❌ Request failed: {e}")
        print("This could be a network connectivity issue")
    except Exception as e:
        print(f"❌ Unexpected error: {e}")

debug_test_scraping()

Testing URL: https://httpbin.org/html
Status Code: 200
Response length: 3739 characters
✅ Success!


In [6]:
# Try a more reliable test URL
def test_with_different_url():
    # Let's try Google (very reliable)
    url = "https://www.google.com"
    print(f"Testing URL: {url}")

    try:
        response = requests.get(url, timeout=15)  # Longer timeout
        print(f"Status Code: {response.status_code}")

        if response.status_code == 200:
            print("✅ Internet connection working!")
            return True
        else:
            print(f"❌ HTTP Error: {response.status_code}")
            return False

    except Exception as e:
        print(f"❌ Request failed: {e}")
        return False

test_with_different_url()

Testing URL: https://www.google.com
Status Code: 200
✅ Internet connection working!


True

In [7]:
# Test accessing startup directory sites
def check_startup_sites():
    sites_to_test = [
        ("https://www.techinasia.com/", "Tech in Asia"),
        ("https://e27.co/", "e27"),
        ("https://www.crunchbase.com/", "Crunchbase")
    ]

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    accessible_sites = []

    for url, name in sites_to_test:
        try:
            print(f"Testing {name}...")
            response = requests.get(url, headers=headers, timeout=15)

            if response.status_code == 200:
                print(f"✅ {name}: Accessible")
                accessible_sites.append((url, name))
            else:
                print(f"❌ {name}: Status {response.status_code}")

        except Exception as e:
            print(f"❌ {name}: {str(e)}")

        time.sleep(2)  # Be polite, wait between requests

    print(f"\nAccessible sites: {len(accessible_sites)}")
    return accessible_sites

accessible_sites = check_startup_sites()

Testing Tech in Asia...
✅ Tech in Asia: Accessible
Testing e27...
✅ e27: Accessible
Testing Crunchbase...
✅ Crunchbase: Accessible

Accessible sites: 3


In [10]:
# Let's explore Tech in Asia to understand its structure
def explore_techinasia():
    url = "https://www.techinasia.com/"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.content, 'html.parser')

        print("=== Tech in Asia Exploration ===")
        print(f"Page title: {soup.title.string if soup.title else 'No title'}")
        print()

        # Look for startup/company related sections
        print("Looking for startup-related links...")
        links = soup.find_all('a', href=True)

        startup_keywords = ['startup', 'company', 'funding', 'investment', 'directory']
        relevant_links = []

        for link in links[:50]:  # Check first 50 links
            href = link.get('href', '').lower()
            text = link.get_text().strip().lower()

            if any(keyword in href or keyword in text for keyword in startup_keywords):
                if href.startswith('http') or href.startswith('/'):
                    relevant_links.append((text[:50], href))

        print("Found relevant links:")
        for text, href in relevant_links[:10]:  # Show first 10
            print(f"- {text}: {href}")

        return soup

    except Exception as e:
        print(f"Error exploring Tech in Asia: {e}")
        return None

soup = explore_techinasia()

=== Tech in Asia Exploration ===
Page title: Tech in Asia - Connecting Asia's startup ecosystem

Looking for startup-related links...
Found relevant links:
